In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)

# Create the widget once; retain its current value when rerunning the cell.
if "batch_id" not in dbutils.widgets.getAll():
    dbutils.widgets.text(
        "batch_id",
        "full-sample-20261010-01",
        "Bronze batch ID",
    )

batch_id = dbutils.widgets.get("batch_id").strip()
if not batch_id:
    raise ValueError("batch_id is required")

bronze_table = "workspace.tmdb_lakehouse.bronze_movie_records"
bronze_batch_df = (
    spark.table(bronze_table)
    .filter(F.col("batch_id") == batch_id)
)

bronze_count = bronze_batch_df.count()
if bronze_count == 0:
    raise ValueError(f"No Bronze records found for batch_id={batch_id}")

print(f"Bronze rows in selected batch: {bronze_count}")
bronze_batch_df.select(
    "batch_id",
    "load_type",
    "source_extracted_at",
).distinct().show(truncate=False)

child_movie_schema = StructType([
    StructField("id", LongType(), True),

    StructField("genres", ArrayType(StructType([
        StructField("id", LongType(), True),
        StructField("name", StringType(), True),
    ])), True),

    StructField("production_companies", ArrayType(StructType([
        StructField("id", LongType(), True),
        StructField("name", StringType(), True),
        StructField("origin_country", StringType(), True),
    ])), True),

    StructField("credits", StructType([
        StructField("cast", ArrayType(StructType([
            StructField("id", LongType(), True),
            StructField("credit_id", StringType(), True),
            StructField("name", StringType(), True),
            StructField("character", StringType(), True),
            StructField("order", IntegerType(), True),
        ])), True),
    ]), True),

    StructField("keywords", StructType([
        StructField("keywords", ArrayType(StructType([
            StructField("id", LongType(), True),
            StructField("name", StringType(), True),
        ])), True),
    ]), True),
])

parsed_child_movies_df = (
    bronze_batch_df
    .select(
        "movie_id",
        "batch_id",
        "load_type",
        "source_file",
        "source_extracted_at",
        F.from_json("raw_payload", child_movie_schema).alias("movie"),
    )
    .select(
        "movie_id",
        "batch_id",
        "load_type",
        "source_file",
        "source_extracted_at",
        "movie.*",
    )
)

parsed_count = parsed_child_movies_df.count()
bad_ids = parsed_child_movies_df.filter(
    F.col("id").isNull() | (F.col("id") != F.col("movie_id"))
).count()

print(f"Parsed movie rows: {parsed_count}")
print(f"Missing or mismatched parsed IDs: {bad_ids}")
parsed_child_movies_df.select(
    "movie_id",
    "id",
    F.size("genres").alias("genre_count"),
    F.size("production_companies").alias("studio_count"),
    F.size("credits.cast").alias("cast_count"),
    F.size("keywords.keywords").alias("keyword_count"),
).show(10, truncate=False)

if bad_ids:
    raise ValueError(
        f"{bad_ids} Bronze records have a missing or mismatched movie ID"
    )

In [0]:
from pyspark.sql import functions as F

# A missing collection is excluded; a present empty array remains a snapshot candidate.
genre_snapshot_candidates_df = (
    parsed_child_movies_df
    .filter(F.col("genres").isNotNull())
    .select(
        "movie_id",
        "source_extracted_at",
        F.col("batch_id").alias("source_batch_id"),
        F.size("genres").alias("genre_count"),
    )
)

silver_genres_source_df = (
    parsed_child_movies_df
    .filter(F.col("genres").isNotNull())
    .select(
        "movie_id",
        "batch_id",
        "source_file",
        "source_extracted_at",
        F.explode("genres").alias("genre"),
    )
    .select(
        "movie_id",
        F.col("genre.id").alias("genre_id"),
        F.col("genre.name").alias("genre_name"),
        F.col("batch_id").alias("source_batch_id"),
        "source_file",
        "source_extracted_at",
    )
)

invalid_genres_df = silver_genres_source_df.filter(
    F.col("movie_id").isNull()
    | F.col("genre_id").isNull()
    | F.col("genre_name").isNull()
)

invalid_genre_count = invalid_genres_df.count()
if invalid_genre_count:
    display(invalid_genres_df.limit(20))
    raise ValueError(
        f"Found {invalid_genre_count} genre rows missing a required field"
    )

# Identical movie-genre entries can be collapsed; conflicting names must stop processing.
conflicting_genres_df = (
    silver_genres_source_df
    .groupBy("movie_id", "genre_id")
    .agg(F.countDistinct("genre_name").alias("name_versions"))
    .filter(F.col("name_versions") > 1)
)

conflicting_genre_count = conflicting_genres_df.count()
if conflicting_genre_count:
    display(conflicting_genres_df.limit(20))
    raise ValueError(
        f"Found {conflicting_genre_count} movie-genre keys with conflicting names"
    )

silver_genres_source_df = (
    silver_genres_source_df
    .dropDuplicates(["movie_id", "genre_id"])
    .withColumn(
        "row_hash",
        F.sha2(
            F.to_json(F.struct("movie_id", "genre_id", "genre_name")),
            256,
        ),
    )
    .withColumn("load_timestamp", F.current_timestamp())
)

duplicate_genre_keys = (
    silver_genres_source_df
    .groupBy("movie_id", "genre_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Movies with a present genre collection:", genre_snapshot_candidates_df.count())
print(
    "Movies with an empty genre collection:",
    genre_snapshot_candidates_df.filter(F.col("genre_count") == 0).count(),
)
print(
    "Movies with a missing or unparsed genre collection:",
    parsed_child_movies_df.filter(F.col("genres").isNull()).count(),
)
print("Silver genre rows prepared:", silver_genres_source_df.count())
print("Invalid genre rows:", invalid_genre_count)
print("Conflicting genre keys:", conflicting_genre_count)
print("Duplicate movie-genre keys after preparation:", duplicate_genre_keys)

silver_genres_source_df.printSchema()
display(silver_genres_source_df.limit(10))

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql.types import (
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

logs_table = "workspace.tmdb_lakehouse.pipeline_execution_logs"

child_log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("load_type", StringType(), False),
    StructField("input_parameter", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("rows_inserted", LongType(), False),
    StructField("rows_updated", LongType(), False),
    StructField("rows_quarantined", LongType(), False),
    StructField("error_details", StringType(), True),
    StructField("load_timestamp", TimestampType(), False),
])

def write_child_run_log(
    run_id,
    started_at,
    collection_name,
    status,
    rows_inserted,
    rows_updated,
    rows_quarantined=0,
    error_details=None,
):
    finished_at = datetime.now(timezone.utc).replace(tzinfo=None)
    load_type = parsed_child_movies_df.select("load_type").first()["load_type"]

    log_row = [(
        run_id,
        "Bronze-to-Silver",
        load_type,
        f"batch_id={batch_id}; collection={collection_name}",
        batch_id,
        started_at,
        finished_at,
        status,
        rows_inserted,
        rows_updated,
        rows_quarantined,
        error_details,
        finished_at,
    )]

    (
        spark.createDataFrame(log_row, schema=child_log_schema)
        .write.format("delta")
        .mode("append")
        .saveAsTable(logs_table)
    )

print("Child-table audit helper defined.")
def merge_child_collection(
    collection_name,
    table_name,
    snapshot_candidates_df,
    source_df,
    child_key,
):
    state_table = "workspace.tmdb_lakehouse.silver_child_snapshot_state"
    key_columns = ["movie_id", child_key]
    run_id = str(uuid4())
    started_at = datetime.now(timezone.utc).replace(tzinfo=None)
    rows_inserted = rows_updated = rows_deleted = rows_quarantined = 0

    try:
        duplicate_snapshots = (
            snapshot_candidates_df.groupBy("movie_id").count()
            .filter(F.col("count") > 1).count()
        )
        if duplicate_snapshots:
            raise ValueError(
                f"Found {duplicate_snapshots} duplicate {collection_name} snapshots"
            )

        duplicate_keys = (
            source_df.groupBy(*key_columns).count()
            .filter(F.col("count") > 1).count()
        )
        if duplicate_keys:
            raise ValueError(
                f"Found {duplicate_keys} duplicate keys in the {collection_name} source"
            )

        stored_state_df = (
            spark.table(state_table)
            .filter(F.col("collection_name") == collection_name)
            .select(
                "movie_id",
                F.col("source_extracted_at").alias("stored_extracted_at"),
                F.col("source_batch_id").alias("stored_batch_id"),
            )
        )
        eligible_snapshots_df = (
            snapshot_candidates_df.alias("incoming")
            .join(stored_state_df.alias("stored"), on="movie_id", how="left")
            .filter(
                F.col("stored.stored_batch_id").isNull()
                | (F.col("incoming.source_extracted_at") > F.col("stored.stored_extracted_at"))
                | (
                    (F.col("incoming.source_extracted_at") == F.col("stored.stored_extracted_at"))
                    & (F.col("incoming.source_batch_id") == F.col("stored.stored_batch_id"))
                )
            )
            .select("movie_id", "source_extracted_at", "source_batch_id")
        )

        candidate_count = snapshot_candidates_df.count()
        eligible_count = eligible_snapshots_df.count()
        print(f"Eligible {collection_name} movie snapshots:", eligible_count)
        if eligible_count != candidate_count:
            raise ValueError(
                f"Some {collection_name} snapshots are older than stored data "
                "or have an ambiguous timestamp. Resolve snapshot ordering before processing."
            )

        eligible_ids_df = eligible_snapshots_df.select("movie_id").distinct()
        eligible_source_df = source_df.join(eligible_ids_df, on="movie_id", how="inner")
        existing_df = spark.table(table_name)
        missing_columns = sorted(set(existing_df.columns) - set(eligible_source_df.columns))
        extra_columns = sorted(set(eligible_source_df.columns) - set(existing_df.columns))
        if missing_columns or extra_columns:
            raise ValueError(
                f"{collection_name} source/table columns differ; "
                f"missing={missing_columns}, extra={extra_columns}"
            )

        rows_inserted = eligible_source_df.join(
            existing_df, key_columns, "left_anti"
        ).count()
        rows_updated = (
            eligible_source_df.alias("incoming")
            .join(
                existing_df.alias("stored"),
                (F.col(f"incoming.{key_columns[0]}") == F.col(f"stored.{key_columns[0]}"))
                & (F.col(f"incoming.{key_columns[1]}") == F.col(f"stored.{key_columns[1]}")),
                "inner",
            )
            .filter(
                (~F.col("incoming.row_hash").eqNullSafe(F.col("stored.row_hash")))
                | (~F.col("incoming.source_extracted_at").eqNullSafe(F.col("stored.source_extracted_at")))
                | (~F.col("incoming.source_batch_id").eqNullSafe(F.col("stored.source_batch_id")))
                | (~F.col("incoming.source_file").eqNullSafe(F.col("stored.source_file")))
            )
            .count()
        )
        stale_rows_df = (
            existing_df.join(eligible_ids_df, on="movie_id", how="inner")
            .join(eligible_source_df.select(*key_columns), on=key_columns, how="left_anti")
            .select(*key_columns).dropDuplicates()
        )
        rows_deleted = stale_rows_df.count()

        suffix = run_id.replace("-", "")
        source_view = f"_{collection_name}_source_{suffix}"
        stale_view = f"_{collection_name}_stale_{suffix}"
        state_view = f"_{collection_name}_state_{suffix}"
        eligible_source_df.createOrReplaceTempView(source_view)
        stale_rows_df.createOrReplaceTempView(stale_view)

        quoted_keys = [f"`{column}`" for column in key_columns]
        merge_condition = " AND ".join(
            f"target.{key} = source.{key}" for key in quoted_keys
        )
        changed_condition = " OR ".join([
            "NOT (target.`row_hash` <=> source.`row_hash`)",
            "NOT (target.`source_extracted_at` <=> source.`source_extracted_at`)",
            "NOT (target.`source_batch_id` <=> source.`source_batch_id`)",
            "NOT (target.`source_file` <=> source.`source_file`)",
        ])
        assignments = [
            (
                "target.`load_timestamp` = CASE "
                "WHEN NOT (target.`row_hash` <=> source.`row_hash`) "
                "THEN source.`load_timestamp` ELSE target.`load_timestamp` END"
            ) if column == "load_timestamp" else
            f"target.`{column}` = source.`{column}`"
            for column in existing_df.columns
        ]
        insert_columns = ", ".join(f"`{column}`" for column in existing_df.columns)
        insert_values = ", ".join(f"source.`{column}`" for column in existing_df.columns)

        spark.sql(f"""
            MERGE INTO {table_name} AS target
            USING {source_view} AS source
            ON {merge_condition}
            WHEN MATCHED AND ({changed_condition})
            THEN UPDATE SET {', '.join(assignments)}
            WHEN NOT MATCHED THEN INSERT ({insert_columns})
            VALUES ({insert_values})
        """)
        spark.sql(f"""
            MERGE INTO {table_name} AS target
            USING {stale_view} AS stale
            ON target.{quoted_keys[0]} = stale.{quoted_keys[0]}
               AND target.{quoted_keys[1]} = stale.{quoted_keys[1]}
            WHEN MATCHED THEN DELETE
        """)

        eligible_snapshots_df.select(
            "movie_id",
            F.lit(collection_name).alias("collection_name"),
            "source_extracted_at",
            "source_batch_id",
            F.current_timestamp().alias("load_timestamp"),
        ).createOrReplaceTempView(state_view)
        spark.sql(f"""
            MERGE INTO {state_table} AS target
            USING {state_view} AS source
            ON target.movie_id = source.movie_id
               AND target.collection_name = source.collection_name
            WHEN MATCHED AND source.source_extracted_at > target.source_extracted_at
            THEN UPDATE SET
                target.source_extracted_at = source.source_extracted_at,
                target.source_batch_id = source.source_batch_id,
                target.load_timestamp = source.load_timestamp
            WHEN NOT MATCHED THEN INSERT (
                movie_id, collection_name, source_extracted_at,
                source_batch_id, load_timestamp
            ) VALUES (
                source.movie_id, source.collection_name, source.source_extracted_at,
                source.source_batch_id, source.load_timestamp
            )
        """)

        write_child_run_log(
            run_id, started_at, collection_name, "SUCCESS",
            rows_inserted, rows_updated, rows_quarantined,
        )
        print(f"{collection_name.title()} rows inserted:", rows_inserted)
        print(f"{collection_name.title()} rows updated:", rows_updated)
        print(f"Stale {collection_name} rows deleted:", rows_deleted)
        print(f"{collection_name.title()} rows now stored:", spark.table(table_name).count())
        return {
            "inserted": rows_inserted,
            "updated": rows_updated,
            "deleted": rows_deleted,
            "quarantined": rows_quarantined,
        }
    except Exception as exc:
        try:
            write_child_run_log(
                run_id, started_at, collection_name, "FAILURE",
                rows_inserted, rows_updated, rows_quarantined, str(exc)[:4000],
            )
        except Exception as log_exc:
            raise RuntimeError(
                f"{collection_name.title()} processing failed: {exc}; "
                f"failure logging also failed: {log_exc}"
            ) from log_exc
        raise

In [ ]:
merge_child_collection(
    collection_name="genres",
    table_name="workspace.tmdb_lakehouse.silver_movie_genres",
    snapshot_candidates_df=genre_snapshot_candidates_df,
    source_df=silver_genres_source_df,
    child_key="genre_id",
)

In [0]:
spark.sql("""
SELECT run_id, load_type, batch_id, status,
       rows_inserted, rows_updated, rows_quarantined, error_details
FROM workspace.tmdb_lakehouse.pipeline_execution_logs
WHERE layer = 'Bronze-to-Silver'
  AND input_parameter LIKE '%collection=genres%'
ORDER BY start_time DESC
LIMIT 5
""").show(truncate=False)

In [0]:
from pyspark.sql import functions as F

# Include present empty arrays as snapshot candidates.
studio_snapshot_candidates_df = (
    parsed_child_movies_df
    .filter(F.col("production_companies").isNotNull())
    .select(
        "movie_id",
        "source_extracted_at",
        F.col("batch_id").alias("source_batch_id"),
        F.size("production_companies").alias("studio_count"),
    )
)

silver_studios_source_df = (
    parsed_child_movies_df
    .filter(F.col("production_companies").isNotNull())
    .select(
        "movie_id",
        "batch_id",
        "source_file",
        "source_extracted_at",
        F.explode("production_companies").alias("studio"),
    )
    .select(
        "movie_id",
        F.col("studio.id").alias("company_id"),
        F.col("studio.name").alias("company_name"),
        F.col("studio.origin_country").alias("origin_country"),
        F.col("batch_id").alias("source_batch_id"),
        "source_file",
        "source_extracted_at",
    )
)

invalid_studios_df = silver_studios_source_df.filter(
    F.col("movie_id").isNull()
    | F.col("company_id").isNull()
    | F.col("company_name").isNull()
)
invalid_studio_count = invalid_studios_df.count()

if invalid_studio_count:
    display(invalid_studios_df.limit(20))
    raise ValueError(
        f"Found {invalid_studio_count} studio rows missing a required field"
    )

# Stop if a movie-company key has conflicting attributes in this batch.
conflicting_studios_df = (
    silver_studios_source_df
    .groupBy("movie_id", "company_id")
    .agg(
        F.countDistinct(
            F.to_json(F.struct("company_name", "origin_country"))
        ).alias("attribute_versions")
    )
    .filter(F.col("attribute_versions") > 1)
)
conflicting_studio_count = conflicting_studios_df.count()

if conflicting_studio_count:
    display(conflicting_studios_df.limit(20))
    raise ValueError(
        f"Found {conflicting_studio_count} movie-company keys with conflicting attributes"
    )

silver_studios_source_df = (
    silver_studios_source_df
    .dropDuplicates(["movie_id", "company_id"])
    .withColumn(
        "row_hash",
        F.sha2(
            F.to_json(
                F.struct(
                    "movie_id",
                    "company_id",
                    "company_name",
                    "origin_country",
                )
            ),
            256,
        ),
    )
    .withColumn("load_timestamp", F.current_timestamp())
)

duplicate_studio_keys = (
    silver_studios_source_df
    .groupBy("movie_id", "company_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Movies with a present studio collection:", studio_snapshot_candidates_df.count())
print(
    "Movies with an empty studio collection:",
    studio_snapshot_candidates_df.filter(F.col("studio_count") == 0).count(),
)
print(
    "Movies with a missing or unparsed studio collection:",
    parsed_child_movies_df.filter(F.col("production_companies").isNull()).count(),
)
print("Silver studio rows prepared:", silver_studios_source_df.count())
print("Invalid studio rows:", invalid_studio_count)
print("Conflicting movie-company keys:", conflicting_studio_count)
print("Duplicate movie-company keys after preparation:", duplicate_studio_keys)

silver_studios_source_df.printSchema()
display(silver_studios_source_df.limit(10))

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb_lakehouse.silver_movie_studios (
    movie_id BIGINT,
    company_id BIGINT,
    company_name STRING,
    origin_country STRING,
    source_batch_id STRING,
    source_file STRING,
    source_extracted_at TIMESTAMP,
    row_hash STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

spark.sql("""
DESCRIBE TABLE workspace.tmdb_lakehouse.silver_movie_studios
""").show(truncate=False)

In [ ]:
merge_child_collection(
    collection_name="studios",
    table_name="workspace.tmdb_lakehouse.silver_movie_studios",
    snapshot_candidates_df=studio_snapshot_candidates_df,
    source_df=silver_studios_source_df,
    child_key="company_id",
)

In [0]:
spark.sql("""
SELECT run_id, batch_id, status, rows_inserted, rows_updated,
       rows_quarantined, error_details
FROM workspace.tmdb_lakehouse.pipeline_execution_logs
WHERE layer = 'Bronze-to-Silver'
  AND input_parameter LIKE '%collection=studios%'
ORDER BY start_time DESC
LIMIT 5
""").show(truncate=False)

In [0]:
from pyspark.sql import functions as F

# Include present empty arrays as snapshots; exclude missing or unparsed collections.
cast_snapshot_candidates_df = (
    parsed_child_movies_df
    .filter(F.col("credits.cast").isNotNull())
    .select(
        "movie_id",
        "source_extracted_at",
        F.col("batch_id").alias("source_batch_id"),
        F.size("credits.cast").alias("cast_count"),
    )
)

silver_cast_source_df = (
    parsed_child_movies_df
    .filter(F.col("credits.cast").isNotNull())
    .select(
        "movie_id",
        "batch_id",
        "source_file",
        "source_extracted_at",
        F.explode("credits.cast").alias("cast_member"),
    )
    .select(
        "movie_id",
        F.col("cast_member.credit_id").alias("credit_id"),
        F.col("cast_member.id").alias("person_id"),
        F.col("cast_member.name").alias("cast_name"),
        F.col("cast_member.character").alias("character"),
        F.col("cast_member.order").alias("cast_order"),
        F.col("batch_id").alias("source_batch_id"),
        "source_file",
        "source_extracted_at",
    )
)

# The logical key is (movie_id, credit_id); other cast attributes may be null.
invalid_cast_df = silver_cast_source_df.filter(
    F.col("movie_id").isNull() | F.col("credit_id").isNull()
)
invalid_cast_count = invalid_cast_df.count()

if invalid_cast_count:
    display(invalid_cast_df.limit(20))
    raise ValueError(
        f"Found {invalid_cast_count} cast rows missing movie_id or credit_id"
    )

# Identical duplicate keys can be collapsed; conflicting attributes must be investigated.
conflicting_cast_df = (
    silver_cast_source_df
    .groupBy("movie_id", "credit_id")
    .agg(
        F.countDistinct(
            F.to_json(
                F.struct("person_id", "cast_name", "character", "cast_order")
            )
        ).alias("attribute_versions")
    )
    .filter(F.col("attribute_versions") > 1)
)
conflicting_cast_count = conflicting_cast_df.count()

if conflicting_cast_count:
    display(conflicting_cast_df.limit(20))
    raise ValueError(
        f"Found {conflicting_cast_count} movie-credit keys with conflicting attributes"
    )

silver_cast_source_df = (
    silver_cast_source_df
    .dropDuplicates(["movie_id", "credit_id"])
    .withColumn(
        "row_hash",
        F.sha2(
            F.to_json(
                F.struct(
                    "movie_id",
                    "credit_id",
                    "person_id",
                    "cast_name",
                    "character",
                    "cast_order",
                )
            ),
            256,
        ),
    )
    .withColumn("load_timestamp", F.current_timestamp())
)

duplicate_cast_keys = (
    silver_cast_source_df
    .groupBy("movie_id", "credit_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Movies with a present cast collection:", cast_snapshot_candidates_df.count())
print(
    "Movies with an empty cast collection:",
    cast_snapshot_candidates_df.filter(F.col("cast_count") == 0).count(),
)
print(
    "Movies with a missing or unparsed cast collection:",
    parsed_child_movies_df.filter(F.col("credits.cast").isNull()).count(),
)
print("Silver cast rows prepared:", silver_cast_source_df.count())
print("Invalid cast keys:", invalid_cast_count)
print("Conflicting movie-credit keys:", conflicting_cast_count)
print("Duplicate movie-credit keys after preparation:", duplicate_cast_keys)

silver_cast_source_df.printSchema()
display(silver_cast_source_df.limit(10))

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb_lakehouse.silver_movie_cast (
    movie_id BIGINT,
    credit_id STRING,
    person_id BIGINT,
    cast_name STRING,
    character STRING,
    cast_order INT,
    source_batch_id STRING,
    source_file STRING,
    source_extracted_at TIMESTAMP,
    row_hash STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

spark.sql("""
DESCRIBE TABLE workspace.tmdb_lakehouse.silver_movie_cast
""").show(truncate=False)

In [ ]:
merge_child_collection(
    collection_name="cast",
    table_name="workspace.tmdb_lakehouse.silver_movie_cast",
    snapshot_candidates_df=cast_snapshot_candidates_df,
    source_df=silver_cast_source_df,
    child_key="credit_id",
)

In [0]:
from pyspark.sql import functions as F

# Include present empty arrays as snapshots; exclude missing or unparsed collections.
keyword_snapshot_candidates_df = (
    parsed_child_movies_df
    .filter(F.col("keywords.keywords").isNotNull())
    .select(
        "movie_id",
        "source_extracted_at",
        F.col("batch_id").alias("source_batch_id"),
        F.size("keywords.keywords").alias("keyword_count"),
    )
)

silver_keywords_source_df = (
    parsed_child_movies_df
    .filter(F.col("keywords.keywords").isNotNull())
    .select(
        "movie_id",
        "batch_id",
        "source_file",
        "source_extracted_at",
        F.explode("keywords.keywords").alias("keyword"),
    )
    .select(
        "movie_id",
        F.col("keyword.id").alias("keyword_id"),
        F.col("keyword.name").alias("keyword_name"),
        F.col("batch_id").alias("source_batch_id"),
        "source_file",
        "source_extracted_at",
    )
)

invalid_keywords_df = silver_keywords_source_df.filter(
    F.col("movie_id").isNull()
    | F.col("keyword_id").isNull()
    | F.col("keyword_name").isNull()
)
invalid_keyword_count = invalid_keywords_df.count()

if invalid_keyword_count:
    display(invalid_keywords_df.limit(20))
    raise ValueError(
        f"Found {invalid_keyword_count} keyword rows missing a required field"
    )

conflicting_keywords_df = (
    silver_keywords_source_df
    .groupBy("movie_id", "keyword_id")
    .agg(F.countDistinct("keyword_name").alias("name_versions"))
    .filter(F.col("name_versions") > 1)
)
conflicting_keyword_count = conflicting_keywords_df.count()

if conflicting_keyword_count:
    display(conflicting_keywords_df.limit(20))
    raise ValueError(
        f"Found {conflicting_keyword_count} movie-keyword keys with conflicting names"
    )

silver_keywords_source_df = (
    silver_keywords_source_df
    .dropDuplicates(["movie_id", "keyword_id"])
    .withColumn(
        "row_hash",
        F.sha2(
            F.to_json(
                F.struct("movie_id", "keyword_id", "keyword_name")
            ),
            256,
        ),
    )
    .withColumn("load_timestamp", F.current_timestamp())
)

duplicate_keyword_keys = (
    silver_keywords_source_df
    .groupBy("movie_id", "keyword_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Movies with a present keyword collection:",
    keyword_snapshot_candidates_df.count(),
)
print(
    "Movies with an empty keyword collection:",
    keyword_snapshot_candidates_df.filter(F.col("keyword_count") == 0).count(),
)
print(
    "Movies with a missing or unparsed keyword collection:",
    parsed_child_movies_df.filter(F.col("keywords.keywords").isNull()).count(),
)
print("Silver keyword rows prepared:", silver_keywords_source_df.count())
print("Invalid keyword rows:", invalid_keyword_count)
print("Conflicting movie-keyword keys:", conflicting_keyword_count)
print("Duplicate movie-keyword keys after preparation:", duplicate_keyword_keys)

silver_keywords_source_df.printSchema()
display(silver_keywords_source_df.limit(10))


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb_lakehouse.silver_movie_keywords (
    movie_id BIGINT,
    keyword_id BIGINT,
    keyword_name STRING,
    source_batch_id STRING,
    source_file STRING,
    source_extracted_at TIMESTAMP,
    row_hash STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

spark.sql("""
DESCRIBE TABLE workspace.tmdb_lakehouse.silver_movie_keywords
""").show(truncate=False)

In [ ]:
merge_child_collection(
    collection_name="keywords",
    table_name="workspace.tmdb_lakehouse.silver_movie_keywords",
    snapshot_candidates_df=keyword_snapshot_candidates_df,
    source_df=silver_keywords_source_df,
    child_key="keyword_id",
)